In [10]:
import numpy as np
import open3d as o3d
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import os
from naming_convention import get_naming_convention

# Initialize naming convention for consistent paths
naming = get_naming_convention()

# 1. Load and visualize point cloud
def visualize_point_cloud(ply_path):
    # Load the PLY file
    pcd = o3d.io.read_point_cloud(ply_path)
    
    # Visualize point cloud
    o3d.visualization.draw_geometries([pcd])
    return pcd

# 2. Convert to mesh and visualize
def visualize_mesh(pcd):
    # Estimate normals
    pcd.estimate_normals()
    
    # Create mesh using Poisson surface reconstruction with better parameters
    mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
        pcd,
        depth=8,          # Controls the resolution of the reconstruction
        width=0,          # Default width parameter
        scale=1.1,        # Slight scaling to ensure complete surface coverage
        linear_fit=False  # Use non-linear fitting
    )
    
    # Remove low density vertices
    vertices_to_remove = densities < np.quantile(densities, 0.1)
    mesh.remove_vertices_by_mask(vertices_to_remove)
    
    # Visualize mesh
    o3d.visualization.draw_geometries([mesh])
    return mesh

# 3. Create volumetric grid representation
def create_volumetric_grid(pcd, voxel_size=0.05):
    # Create voxel grid
    voxel_grid = o3d.geometry.VoxelGrid.create_from_point_cloud(pcd, voxel_size)
    
    # Visualize voxel grid
    o3d.visualization.draw_geometries([voxel_grid])
    return voxel_grid


# Update the main execution section
if __name__ == "__main__":
    # Use consistent naming convention - check multiple possible paths
    possible_paths = [
        naming.file_patterns['diffraction_mesh'],  # Standard path
        "/home/t/ESAM/cwcorg_17frames_pointcloud.ply",  # Alternative
        "/home/t/ESAM/cwcorg_17frames_pointcloud.ply",  # Legacy path
    ]
    
    ply_path = None
    for path in possible_paths:
        if os.path.exists(path):
            ply_path = path
            break
    
    if ply_path is None:
        print("No mesh file found. Please run the pipeline first or check file paths:")
        for path in possible_paths:
            print(f"  - {path}")
        exit(1)
    
    print(f"Using mesh file: {ply_path}")
    
    # 1. Point Cloud Visualization
    print("Visualizing Point Cloud...")
    pcd = visualize_point_cloud(ply_path)
    
    # 2. Mesh Visualization
    print("\nConverting to Mesh...")
    mesh = visualize_mesh(pcd)
    
    # 3. Volumetric Grid Visualization
    print("\nCreating Volumetric Grid...")
    voxel_grid = create_volumetric_grid(pcd)
    

Using mesh file: /home/t/ESAM/cwcorg_17frames_pointcloud.ply
Visualizing Point Cloud...


KeyboardInterrupt: 